# Abliterate Qwen3-4B-Thinking-2507 with Heretic

Part of the **say-no-more** experiment. Runs [Heretic](https://github.com/p-e-w/heretic)
on the exact base model used for the baseline eval (pinned commit `768f209d9ea8…`)
and uploads the abliterated result to your Hugging Face account.

### Before you run
1. **Settings → Accelerator → GPU** (T4 ×2 is fine for a 4B model).
2. **Settings → Internet → On**.
3. **Add-ons → Secrets → add `HF_TOKEN`** (a Hugging Face token with **write** access),
   and make sure it is **Attached** to this notebook.

Then **Run All** — or **Save Version → Save & Run All (Commit)** to run it headless.
Both work: the abliteration runs in a subprocess (see below), so no kernel restart is needed.

### Why a subprocess
Kaggle preloads numpy, and installing Heretic pulls a different numpy + other deps. Importing
Heretic in the notebook kernel then hits a half-swapped numpy (`_slice` / `_blas_supports_fpe`
errors). Restarting fixes it interactively but **Commit mode can't restart**. So the install
pins a compatible numpy and removes Kaggle's stale `torchao`, and the actual run happens in a
**fresh `python` subprocess** that imports everything cleanly from disk.

## 1 · GPU check + install

Pins `numpy` to the 2.2 line (what Heretic requires and what Kaggle's scipy/numba expect) so
the install can't drag in an incompatible numpy, and removes Kaggle's old `torchao` (peft
rejects torchao <0.16 if it is present, and the merge path doesn't use it).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "NO GPU — enable it in Settings → Accelerator"
%pip install -q "numpy>=2.2,<2.3" heretic-llm
%pip uninstall -y torchao

## 2 · Configuration
Edit if you like. Fewer trials = faster but lower-quality abliteration.

In [ ]:
MODEL_ID     = "Qwen/Qwen3-4B-Thinking-2507"
MODEL_COMMIT = "768f209d9ea81521153ed38c47d515654e938aea"   # exact revision used by the baseline
REPO_SUFFIX  = "Qwen3-4B-Thinking-2507-say-no-more"   # -> <your-username>/<this>
PRIVATE      = False   # True to upload as a private repo
N_TRIALS     = 100     # Heretic default is 200; a T4 does ~100 in a few hours
SEED         = 0

## 3 · Hugging Face login (from the `HF_TOKEN` secret)

In [ ]:
import huggingface_hub
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    raise RuntimeError(
        "Could not read the HF_TOKEN secret. Add it under Add-ons -> Secrets "
        "(name: HF_TOKEN, a token with WRITE access), and make sure it is "
        "ATTACHED to this notebook."
    ) from None

huggingface_hub.login(token=HF_TOKEN, add_to_git_credential=False)
USERNAME = huggingface_hub.whoami(token=HF_TOKEN)["name"]
REPO_ID  = f"{USERNAME}/{REPO_SUFFIX}"
print("Will upload to:", REPO_ID, "(private)" if PRIVATE else "(public)")

## 4 · Write the abliteration driver

Heretic's post-run menu is interactive, and in 1.4.0 only `--export-strategy` is a real CLI
flag. This script monkeypatches `questionary` so every prompt is answered automatically
(export=merge, trial=best, action=upload, repo/visibility/reproducibility preset, token from
the environment) and raises a sentinel to stop after a single upload. It reads its settings
from environment variables so the next cell can run it in a clean subprocess.

In [ ]:
%%writefile /kaggle/working/run_heretic.py
import os, sys, questionary, huggingface_hub

MODEL_ID     = os.environ["MODEL_ID"]
MODEL_COMMIT = os.environ.get("MODEL_COMMIT") or None
REPO_ID      = os.environ["REPO_ID"]
PRIVATE      = os.environ.get("PRIVATE", "0") == "1"
N_TRIALS     = os.environ.get("N_TRIALS", "100")
SEED         = os.environ.get("SEED", "0")
HF_TOKEN     = os.environ["HF_TOKEN"]

huggingface_hub.login(token=HF_TOKEN, add_to_git_credential=False)
huggingface_hub.get_token = lambda: HF_TOKEN   # Heretic reads the token from here

class _Done(BaseException):
    """Raised to stop Heretic's menu loop after the single upload."""

_state = {"action_calls": 0}
def _cval(c):
    return getattr(c, "value", c)

def _pick(message, choices=None, default=None):
    msg = str(message).lower()
    if "token" in msg:
        return HF_TOKEN
    if "repository" in msg and "public" not in msg and "private" not in msg:
        return REPO_ID
    if "additional trials" in msg:
        return ""
    if "what do you want to do" in msg:
        _state["action_calls"] += 1
        if _state["action_calls"] == 1:
            for c in (choices or []):
                if str(_cval(c)).lower() == "upload":
                    return _cval(c)
            return _cval(choices[0]) if choices else "upload"
        raise _Done
    if choices:
        if "export" in msg:
            for c in choices:
                if str(_cval(c)).lower() == "merge":
                    return _cval(c)
        if "public or private" in msg:
            want = "private" if PRIVATE else "public"
            for c in choices:
                if want in str(_cval(c)).lower():
                    return _cval(c)
        if "reproducibility" in msg:
            for c in choices:
                if "full" in str(_cval(c)).lower():
                    return _cval(c)
        return _cval(choices[0])
    return default if default is not None else ""

class _Stub:
    def __init__(self, message, choices=None, default=None, **k):
        self._m, self._c, self._d = message, choices, default
    def ask(self, *a, **k):        return _pick(self._m, self._c, self._d)
    def unsafe_ask(self, *a, **k): return _pick(self._m, self._c, self._d)

def _factory(*a, **k):
    message = a[0] if a else k.get("message", "")
    return _Stub(message, choices=k.get("choices"), default=k.get("default"))

for _n in ("select", "text", "password", "path", "rawselect", "autocomplete", "checkbox"):
    if hasattr(questionary, _n):
        setattr(questionary, _n, _factory)
questionary.confirm = lambda *a, **k: type("C", (), {
    "ask": lambda s, *x, **y: True, "unsafe_ask": lambda s, *x, **y: True})()

sys.argv = ["heretic", "--model", MODEL_ID,
            "--export-strategy", "merge", "--n-trials", N_TRIALS, "--seed", SEED]
if MODEL_COMMIT:
    sys.argv += ["--model-commit", MODEL_COMMIT]

from heretic.main import run
try:
    run()
except _Done:
    print("DONE: abliterated model uploaded to https://huggingface.co/" + REPO_ID)

## 5 · Run it (fresh subprocess)

This is the multi-hour cell: it downloads the model, finds the refusal direction, runs the
Optuna search, and uploads. Output streams live. A fresh `python` process imports numpy/scipy
cleanly, so this works the same whether you Run All interactively or commit it headless.

In [ ]:
import os, sys, subprocess
env = {**os.environ,
       "MODEL_ID": MODEL_ID, "MODEL_COMMIT": MODEL_COMMIT, "REPO_ID": REPO_ID,
       "PRIVATE": "1" if PRIVATE else "0", "N_TRIALS": str(N_TRIALS),
       "SEED": str(SEED), "HF_TOKEN": HF_TOKEN}
proc = subprocess.run([sys.executable, "/kaggle/working/run_heretic.py"], env=env)
if proc.returncode != 0:
    raise SystemExit(f"Heretic exited with code {proc.returncode}")

## 6 · Verify the upload

In [ ]:
from huggingface_hub import list_repo_files
print("Uploaded files:")
for f in sorted(list_repo_files(REPO_ID, token=HF_TOKEN)):
    print("  ", f)
print("\nModel: https://huggingface.co/" + REPO_ID)
print("\nNext, on the Mac:")
print(f"  .venv/bin/hf download {REPO_ID} --local-dir models/{REPO_SUFFIX}")
print(f"  .venv/bin/mlx_lm.convert --hf-path models/{REPO_SUFFIX} \\")
print(f"      --mlx-path models/{REPO_SUFFIX}-mlx-8bit -q --q-bits 8")
print(f"  MODEL=models/{REPO_SUFFIX}-mlx-8bit scripts/serve.sh   # then scripts/eval.sh 03-abliterated")